# OER Study — IrO₂ Results Analysis

This notebook post-processes VASP output files for the **Oxygen Evolution Reaction (OER)** study on the IrO₂ (110) slab.

## Reaction Pathway
The OER proceeds through four elementary electrochemical steps (at potential $U$):

| Step | Reaction | Free Energy |
|------|----------|-------------|
| 1 | $\mathrm{H_2O + * \to OH^* + H^+ + e^-}$ | $\Delta G_1$ |
| 2 | $\mathrm{OH^* \to O^* + H^+ + e^-}$ | $\Delta G_2$ |
| 3 | $\mathrm{O^* + H_2O \to OOH^* + H^+ + e^-}$ | $\Delta G_3$ |
| 4 | $\mathrm{OOH^* \to O_2 + * + H^+ + e^-}$ | $\Delta G_4$ |

At the equilibrium potential $U_0 = 1.23$ V the total free-energy change is $4 \times 1.23 = 4.92$ eV.

**Overpotential:** $\eta = \max(\Delta G_1, \Delta G_2, \Delta G_3, \Delta G_4) / e - 1.23$ V

---
**Reference energies required** (run separate VASP jobs and fill in below):
- `E_slab`    : clean IrO₂ slab  
- `E_OH`      : slab + OH*  
- `E_O`       : slab + O*  
- `E_OOH`     : slab + OOH*  
- `E_H2O`     : gas-phase H₂O molecule  
- `E_H2`      : gas-phase H₂ molecule  

In [ ]:
# ─── Standard imports ────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from ase.io import read as ase_read
from ase.visualize.plot import plot_atoms

from pymatgen.io.vasp.outputs import Outcar, Vasprun

print('All imports successful.')

## 1 · Configuration — Paths and Thermodynamic Corrections

In [ ]:
# ─── Paths ───────────────────────────────────────────────────────────────────
OUTPUTS_DIR = Path('outputs')

slab_dir  = OUTPUTS_DIR / 'slab'      # clean IrO2 slab
OH_dir    = OUTPUTS_DIR / 'slab_OH'   # slab + OH*
O_dir     = OUTPUTS_DIR / 'slab_O'    # slab + O*
OOH_dir   = OUTPUTS_DIR / 'slab_OOH'  # slab + OOH*
H2O_dir   = OUTPUTS_DIR / 'H2O'       # gas-phase H2O
H2_dir    = OUTPUTS_DIR / 'H2'        # gas-phase H2

# ─── Thermodynamic constants ─────────────────────────────────────────────────
# Zero-point energies and entropy corrections (literature, Man et al. 2011)
# All values in eV
ZPE_OH_star  =  0.376
ZPE_O_star   =  0.054
ZPE_OOH_star =  0.471
ZPE_H2O      =  0.558
ZPE_H2       =  0.270

T            =  298.15   # K
TS_OH_star   =  0.0      # eV  (adsorbed, rigid)
TS_O_star    =  0.0      # eV
TS_OOH_star  =  0.0      # eV
TS_H2O       =  0.670    # eV  (liquid)
TS_H2        =  0.410    # eV

# Equilibrium potential
U0 = 1.23   # V

print('Configuration loaded.')
print(f'  T       = {T} K')
print(f'  U0(OER) = {U0} V')

## 2 · Parse VASP Energies

In [ ]:
def parse_final_energy(directory: Path, prefer_vasprun: bool = True) -> float:
    """Return the final DFT total energy (eV) from a VASP calculation directory."""
    vasprun_file = directory / 'vasprun.xml'
    outcar_file  = directory / 'OUTCAR'

    if prefer_vasprun and vasprun_file.exists():
        vr = Vasprun(str(vasprun_file), parse_dos=False, parse_eigen=False)
        e  = vr.final_energy
        print(f'  [{directory.name:12s}] {e:.6f} eV  (vasprun.xml)')
        return e

    if outcar_file.exists():
        oc = Outcar(str(outcar_file))
        e  = oc.final_energy
        print(f'  [{directory.name:12s}] {e:.6f} eV  (OUTCAR)')
        return e

    raise FileNotFoundError(
        f'No vasprun.xml or OUTCAR in {directory}.'
    )


print('Parsing DFT energies...')
E_slab = parse_final_energy(slab_dir)
E_OH   = parse_final_energy(OH_dir)
E_O    = parse_final_energy(O_dir)
E_OOH  = parse_final_energy(OOH_dir)
E_H2O  = parse_final_energy(H2O_dir)
E_H2   = parse_final_energy(H2_dir)

## 3 · Calculate OER Free Energies

In [ ]:
# ─── Helper: Gibbs free energy from DFT energy + ZPE + entropy ───────────────
def G(E_dft, ZPE, TS):
    return E_dft + ZPE - TS


# Gibbs free energies (eV)
G_slab  = G(E_slab, 0.0,           0.0      )
G_OH    = G(E_OH,   ZPE_OH_star,   TS_OH_star )
G_O     = G(E_O,    ZPE_O_star,    TS_O_star  )
G_OOH   = G(E_OOH,  ZPE_OOH_star,  TS_OOH_star)
G_H2O   = G(E_H2O,  ZPE_H2O,       TS_H2O     )
G_H2    = G(E_H2,   ZPE_H2,        TS_H2      )

# Proton–electron free energy at U = 0 (standard hydrogen electrode)
# G(H⁺ + e⁻) = ½ G(H₂) − eU
def G_H_plus_e(U):
    return 0.5 * G_H2 - U


# ─── Free-energy changes at U = 0 V ──────────────────────────────────────────
def calc_delta_G(U=0.0):
    """Return ΔG₁–ΔG₄ (eV) for the four OER steps at electrode potential U."""
    # Step 1: H2O + * → OH* + (H⁺ + e⁻)
    dG1 = G_OH - G_slab - G_H2O + G_H_plus_e(U)
    # Step 2: OH* → O* + (H⁺ + e⁻)
    dG2 = G_O - G_OH + G_H_plus_e(U)
    # Step 3: O* + H2O → OOH* + (H⁺ + e⁻)
    dG3 = G_OOH - G_O - G_H2O + G_H_plus_e(U)
    # Step 4: OOH* → O2(g) + * + (H⁺ + e⁻)
    # G(O₂) constrained so that G(H₂O) - G(H₂) - ½G(O₂) = -2.46 eV (exp)
    G_O2 = 2 * (G_H2O - G_H2 + 2.46)   # constrained so that G(H₂O) - G(H₂) - ½G(O₂) = -2.46 eV (exp.)
    dG4  = G_slab + 0.5 * G_O2 - G_OOH + G_H_plus_e(U)
    return dG1, dG2, dG3, dG4


dG1, dG2, dG3, dG4 = calc_delta_G(U=0.0)

print('OER free-energy changes at U = 0 V:')
print(f'  ΔG₁ (H₂O → OH*)   = {dG1:+.4f} eV')
print(f'  ΔG₂ (OH* → O*)    = {dG2:+.4f} eV')
print(f'  ΔG₃ (O* → OOH*)   = {dG3:+.4f} eV')
print(f'  ΔG₄ (OOH* → O₂)   = {dG4:+.4f} eV')
print(f'  Sum (should ≈ 4.92 eV): {dG1+dG2+dG3+dG4:.4f} eV')

## 4 · Overpotential Calculation

In [ ]:
delta_Gs = [dG1, dG2, dG3, dG4]
step_names = ['ΔG₁', 'ΔG₂', 'ΔG₃', 'ΔG₄']

# Rate-determining step (largest ΔG)
rds_idx      = int(np.argmax(delta_Gs))
G_rds        = delta_Gs[rds_idx]

# Limiting potential
U_limiting   = G_rds     # eV, equals V for 1-electron steps

# Overpotential
eta          = U_limiting - U0

print(f'Rate-determining step : Step {rds_idx + 1} ({step_names[rds_idx]})  —  {G_rds:+.4f} eV')
print(f'Limiting potential    : U_L = {U_limiting:+.4f} V')
print(f'Overpotential         : η   = {eta:+.4f} V')

if eta < 0.4:
    print('  ✔  η < 0.4 V — IrO₂ is an excellent OER catalyst.')
else:
    print('  ✘  η ≥ 0.4 V — significant overpotential observed.')

## 5 · Free-Energy Diagram

In [ ]:
def make_fed(U, ax, color, label):
    """Plot free-energy diagram (FED) at a given potential U."""
    dGs  = list(calc_delta_G(U))
    # Cumulative energies (reaction coordinate)
    Gcum = np.concatenate([[0.0], np.cumsum(dGs)])
    x    = np.arange(len(Gcum))

    bar_w = 0.30
    for xi, gi in zip(x, Gcum):
        ax.hlines(gi, xi - bar_w / 2, xi + bar_w / 2, color=color, linewidth=2.5)
    for i in range(len(x) - 1):
        ax.plot([x[i] + bar_w / 2, x[i + 1] - bar_w / 2],
                [Gcum[i], Gcum[i + 1]], '--', color=color, linewidth=1, alpha=0.6)
    ax.plot([], [], color=color, linewidth=2, label=label)  # legend handle


state_labels = ['*', 'OH*', 'O*', 'OOH*', 'O₂ + *']

fig, ax = plt.subplots(figsize=(8, 5))

# Plot at U = 0 V and at the equilibrium potential U₀ = 1.23 V
make_fed(0.0,  ax, color='steelblue',  label='U = 0 V')
make_fed(U0,   ax, color='forestgreen', label=f'U = U₀ = {U0} V')
make_fed(U_limiting, ax, color='firebrick', label=f'U = U_L = {U_limiting:.2f} V')

ax.axhline(0, color='gray', linewidth=0.7, linestyle=':')
ax.set_xticks(range(len(state_labels)))
ax.set_xticklabels(state_labels, fontsize=11)
ax.set_ylabel('Gibbs Free Energy (eV)', fontsize=11)
ax.set_title('OER Free-Energy Diagram — IrO₂ (110)', fontsize=12)
ax.legend(fontsize=10)

# Annotate overpotential
ax.annotate(f'η = {eta:.2f} V',
            xy=(0.02, 0.92), xycoords='axes fraction',
            fontsize=11, color='firebrick',
            bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='firebrick'))

plt.tight_layout()
plt.savefig('OER_free_energy_diagram.png', dpi=150)
plt.show()
print('Figure saved: OER_free_energy_diagram.png')

## 6 · Visualise the Relaxed IrO₂ Surface

In [ ]:
contcar_path = slab_dir / 'CONTCAR'

if contcar_path.exists():
    atoms = ase_read(str(contcar_path))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    plot_atoms(atoms, axes[0], rotation=('0x,0y,0z'))
    axes[0].set_title('IrO₂ (110) Slab — Top View', fontsize=11)
    axes[0].axis('off')

    plot_atoms(atoms, axes[1], rotation=('90x,0y,0z'))
    axes[1].set_title('IrO₂ (110) Slab — Side View', fontsize=11)
    axes[1].axis('off')

    legend_elements = [
        mpatches.Patch(color='#A8A8A8', label='Ir'),
        mpatches.Patch(color='#FF2222', label='O'),
    ]
    fig.legend(handles=legend_elements, loc='lower center',
               ncol=2, fontsize=10, frameon=False)

    plt.tight_layout()
    plt.savefig('IrO2_slab_structure.png', dpi=150)
    plt.show()
    print('Figure saved: IrO2_slab_structure.png')
else:
    print(f'CONTCAR not found at {contcar_path}.  '
          'Copy CONTCAR to outputs/slab/ after running VASP.')

## 7 · DOS Analysis — Ir d-Band Centre

In [ ]:
vasprun_file = slab_dir / 'vasprun.xml'

if vasprun_file.exists():
    from pymatgen.electronic_structure.plotter import DosPlotter
    from pymatgen.electronic_structure.core import OrbitalType

    vr           = Vasprun(str(vasprun_file), parse_dos=True)
    complete_dos = vr.complete_dos
    efermi       = complete_dos.efermi

    plotter = DosPlotter(sigma=0.05)
    plotter.add_dos('Total DOS', complete_dos)

    # Ir d-band
    from pymatgen.core.periodic_table import Element
    ir_sites = [s for s in complete_dos.structure
                if s.specie == Element('Ir')]

    if ir_sites:
        ir_dos = complete_dos.get_site_spd_dos(ir_sites[0])
        ir_d   = ir_dos[OrbitalType.d]
        plotter.add_dos('Ir d-band', ir_d)

        # d-band centre ε_d
        energies  = np.array(ir_d.energies) - efermi
        densities = np.array(ir_d.get_densities())
        eps_d     = np.trapz(energies * densities, energies) / np.trapz(densities, energies)
        print(f'  Ir d-band centre: ε_d = {eps_d:.3f} eV (relative to E_F)')

    fig = plotter.get_plot(xlim=(-8, 4), ylim=(-30, 30))
    fig.suptitle('Projected DOS — IrO₂ (110) slab', fontsize=12)
    plt.tight_layout()
    plt.savefig('IrO2_dos.png', dpi=150)
    plt.show()
    print('Figure saved: IrO2_dos.png')
else:
    print(f'vasprun.xml not found at {vasprun_file}.  '
          'Run VASP with LORBIT=11 and copy vasprun.xml to outputs/slab/ first.')

## 8 · Summary Table

In [ ]:
import pandas as pd

results = pd.DataFrame({
    'Step': ['H₂O→OH*', 'OH*→O*', 'O*→OOH*', 'OOH*→O₂'],
    'ΔG (eV) @ U=0': [dG1, dG2, dG3, dG4],
    'RDS?': [i == rds_idx for i in range(4)],
})

print(results.to_string(index=False))
print()
print(f'Overpotential η = {eta:.4f} V')